# VULGARIS v0.7.0 — Causal Graph Discovery
## SWaT-Style Water Treatment Plant Demo

**Kaggle free CPU · ~20 min · `pip install vulgaris`**

**What only VULGARIS can do:** learn the causal DAG from unlabelled sensor data
then propagate failure signals through it for root-cause attribution.

- Synthetic SWaT-style data with **known ground-truth causal structure**
- Compare **learned CRG graph vs physical pipeline**
- **Failure propagation**: inject a pump fault → trace downstream effects
- DLinear / LSTM cannot do any of this

In [ ]:
!pip install vulgaris plotly networkx -q

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import networkx as nx
import time, warnings
warnings.filterwarnings('ignore')

import vulgaris
from vulgaris import Vulgaris, ModelConfig, Tensor
from vulgaris import SpectralAdamW, CosineSchedule, VulgarisLoss, TrainingPipeline
from vulgaris.config import ASEConfig, SSSRConfig, CRGConfig, RMCConfig, TrainingConfig, HMBConfig

print('VULGARIS', vulgaris.__version__)
print('NetworkX ', nx.__version__)

## 1. Define SWaT Physical Causal Structure

A 4-stage water treatment plant with 12 sensors.
Each stage: **Flow In → Tank Level → Pump → Flow Out (next stage)**

In [ ]:
# 12 sensors: 4 stages × (flow-in, level, pump)
SENSORS = [
    'FIT101','LIT101','P101',   # Stage 1: raw water intake
    'FIT201','LIT201','P201',   # Stage 2: chemical dosing
    'FIT301','LIT301','P301',   # Stage 3: ultrafiltration
    'FIT401','LIT401','P401',   # Stage 4: reverse osmosis
]
N = len(SENSORS)   # 12

# Ground-truth causal edges (source → target)
GROUND_TRUTH_EDGES = [
    ('FIT101','LIT101'),  # inlet flow fills tank 1
    ('LIT101','P101'),    # level triggers pump 1
    ('P101','FIT201'),    # pump drives flow to stage 2
    ('FIT201','LIT201'),
    ('LIT201','P201'),
    ('P201','FIT301'),
    ('FIT301','LIT301'),
    ('LIT301','P301'),
    ('P301','FIT401'),
    ('FIT401','LIT401'),
    ('LIT401','P401'),
]

# Build ground-truth adjacency matrix
GT = np.zeros((N, N), dtype=np.float32)
for src, dst in GROUND_TRUTH_EDGES:
    GT[SENSORS.index(src), SENSORS.index(dst)] = 1.0

# Node types for colouring
NODE_TYPE = {s: 'flow' if s.startswith('F') else
               'level' if s.startswith('L') else 'pump'
             for s in SENSORS}
TYPE_COLOR = {'flow': '#00D4FF', 'level': '#51CF66', 'pump': '#FF6B6B'}
print('Ground truth edges:', len(GROUND_TRUTH_EDGES))
print('Sensors:', SENSORS)

## 2. Generate SWaT-Style Sensor Data

Data generated with **real causal dependencies** — each sensor value
is a function of its causal parents, so the CRG has something real to learn.

In [ ]:
def generate_swat(T=10000, seed=42):
    rng  = np.random.default_rng(seed)
    data = np.zeros((T, N), dtype=np.float32)
    # FIT101: external inlet flow (sinusoidal + noise)
    data[:, 0] = 0.5 + 0.3*np.sin(2*np.pi*np.arange(T)/500) + rng.normal(0,.05,T)
    for t in range(1, T):
        fit101 = float(data[t, 0])
        # Stage 1
        data[t,1] = max(0, data[t-1,1] + 0.1*fit101 - 0.08*data[t-1,2] + rng.normal(0,.02))
        data[t,2] = float(1/(1+np.exp(-8*(data[t-1,1]-0.5)))) + rng.normal(0,.02)
        p101 = float(data[t,2])
        # Stage 2
        data[t,3] = 0.9*p101 + rng.normal(0,.03)
        data[t,4] = max(0, data[t-1,4] + 0.1*data[t,3] - 0.08*data[t-1,5] + rng.normal(0,.02))
        data[t,5] = float(1/(1+np.exp(-8*(data[t-1,4]-0.5)))) + rng.normal(0,.02)
        p201 = float(data[t,5])
        # Stage 3
        data[t,6] = 0.85*p201 + rng.normal(0,.03)
        data[t,7] = max(0, data[t-1,7] + 0.1*data[t,6] - 0.08*data[t-1,8] + rng.normal(0,.02))
        data[t,8] = float(1/(1+np.exp(-8*(data[t-1,7]-0.5)))) + rng.normal(0,.02)
        p301 = float(data[t,8])
        # Stage 4
        data[t,9]  = 0.80*p301 + rng.normal(0,.03)
        data[t,10] = max(0, data[t-1,10] + 0.1*data[t,9] - 0.08*data[t-1,11] + rng.normal(0,.02))
        data[t,11] = float(1/(1+np.exp(-8*(data[t-1,10]-0.5)))) + rng.normal(0,.02)
    return data

data = generate_swat(T=12000)
T_tr = int(len(data)*0.7)
tr, te = data[:T_tr], data[T_tr:]
print(f'Train: {tr.shape}  Test: {te.shape}')

# Plot first 500 steps
fig = make_subplots(3, 1, subplot_titles=['Flow sensors','Level sensors','Pump outputs'])
x = list(range(500))
for i,s in enumerate(SENSORS):
    nt = NODE_TYPE[s]
    row = {'flow':1,'level':2,'pump':3}[nt]
    fig.add_trace(go.Scatter(x=x, y=data[:500,i].tolist(), name=s,
        line=dict(color=TYPE_COLOR[nt], width=1.2), showlegend=True),
        row=row, col=1)
fig.update_layout(template='plotly_dark', height=550,
    title='SWaT-Style Sensor Data — Causal Dependencies Active')
fig.show()

## 3. Train VULGARIS

In [ ]:
from sklearn.preprocessing import StandardScaler

SEQ = 64
sc  = StandardScaler()
tr_n = sc.fit_transform(tr).astype('float32')

D = 64
cfg = ModelConfig(
    input_dim=N, output_dim=N, n_classes=0,
    ase     = ASEConfig(n_filters=8, n_scales=3, filter_len=32, latent_dim=D),
    sssr    = SSSRConfig(state_dim=D, n_heads=4, d_inner=D*2),
    crg     = CRGConfig(n_nodes=N, n_lags=5, dag_lambda=0.01,
                        sparsity_lambda=0.1, update_interval=50),
    rmc     = RMCConfig(n_experts=4),
    hmb     = HMBConfig(embed_dim=D, compress_dim=D//2),
    training= TrainingConfig(lr=5e-4, batch_size=32, seq_len=SEQ,
                             warmup_steps=100, max_steps=5000,
                             gamma_crg=0.001, grad_clip=1.0),
)

model  = Vulgaris(cfg)
opt    = SpectralAdamW(model.parameters(), lr=5e-4)
sched  = CosineSchedule(opt, warmup_steps=100, max_steps=5000, min_lr=1e-5)
loss_f = VulgarisLoss(cfg)
pipe   = TrainingPipeline(model, cfg, loss_f, opt, sched)
print(f'Parameters: {sum(p.data.size for p in model.parameters()):,}')

In [ ]:
B, EPOCHS = 32, 20
t0, log  = time.time(), []

def make_windows(d, seq):
    return np.array([d[i:i+seq] for i in range(len(d)-seq)], 'float32')

X = make_windows(tr_n, SEQ)           # (N, SEQ, C)
y = tr_n[SEQ:, :]                     # (N, C) next step

model.train()
for ep in range(EPOCHS):
    idx = np.random.permutation(len(X))
    el  = 0; nb = 0
    for s in range(0, len(X)-B, B):
        xb = X[idx[s:s+B]].transpose(0,2,1)
        yb = y[idx[s:s+B], 0:1]
        m  = pipe.train_step(xb, yb)
        el += m.get('total_loss', 0); nb += 1
    log.append(el/max(nb,1))
    if (ep+1)%5==0 or ep==0:
        print(f'Epoch {ep+1:2d}  loss={log[-1]:.4f}  {time.time()-t0:.0f}s')

model.eval()

go.Figure(go.Scatter(y=log, mode='lines+markers',
    line=dict(color='#00D4FF', width=2))).update_layout(
    template='plotly_dark', height=260,
    title='Training Loss', xaxis_title='Epoch', yaxis_title='Loss').show()

## 4. Extract Learned Causal Graph

In [ ]:
# Update CRG structure on test data
te_n = sc.transform(te).astype('float32')
X_te = make_windows(te_n, SEQ)

for s in range(0, min(500, len(X_te))-32, 32):
    xb = X_te[s:s+32].transpose(0,2,1)
    model(Tensor(xb))   # forward pass triggers CRG update_structure

# Raw adjacency
W = model.crg.W.data.copy()           # (N, N)

# Apply Neural Granger mask if available
if hasattr(model.crg, 'M'):
    import numpy as _np
    mask = 1 / (1 + _np.exp(-model.crg.M.data))
    W_eff = np.abs(W * mask)
else:
    W_eff = np.abs(W)

# Zero diagonal (no self-loops)
np.fill_diagonal(W_eff, 0)

# Top-K thresholding: keep at most 2x the GT edge count
# This prevents the 'keep 30% of edges = 37 FP' problem
K_max = int(GT.sum()) * 2   # generous upper bound
flat  = W_eff.flatten()
if (flat > 0).sum() > K_max:
    k_thresh = np.sort(flat[flat > 0])[-K_max]
else:
    k_thresh = 0.0
W_thresh = (W_eff >= k_thresh).astype(float) * W_eff
np.fill_diagonal(W_thresh, 0)  # ensure diagonal stays zero

print(f'Ground truth edges  : {int(GT.sum())}')
print(f'Learned edges (top-K): {int((W_thresh > 0).sum())}')

## 5. Graph Visualization — Learned vs Ground Truth

In [ ]:
# Physical layout: 4 stages left→right, 3 sensor rows top→bottom
POS = {}
for i, s in enumerate(SENSORS):
    stage = i // 3
    row   = i % 3
    POS[s] = (stage * 2.5, -row * 1.8)

def draw_graph(adj, title, highlight_gt=False):
    fig = go.Figure()
    # Draw edges
    for i in range(N):
        for j in range(N):
            if adj[i, j] > 0:
                x0, y0 = POS[SENSORS[i]]
                x1, y1 = POS[SENSORS[j]]
                is_gt  = GT[i, j] > 0
                color  = '#00D4FF' if is_gt else '#FF6B6B'
                width  = 1.5 + float(adj[i,j]) * 4
                fig.add_trace(go.Scatter(
                    x=[x0, (x0+x1)/2, x1, None],
                    y=[y0, (y0+y1)/2, y1, None],
                    mode='lines',
                    line=dict(color=color, width=min(width, 6)),
                    hoverinfo='skip', showlegend=False))
                # Arrow head
                fig.add_annotation(
                    x=x1, y=y1, ax=x0, ay=y0,
                    xref='x', yref='y', axref='x', ayref='y',
                    showarrow=True, arrowhead=2, arrowsize=1.2,
                    arrowwidth=max(1, width/2),
                    arrowcolor=color)
    # Draw nodes
    for s in SENSORS:
        x, y = POS[s]
        nt   = NODE_TYPE[s]
        fig.add_trace(go.Scatter(
            x=[x], y=[y], mode='markers+text',
            marker=dict(size=28, color=TYPE_COLOR[nt],
                        line=dict(color='white', width=1.5)),
            text=[s], textposition='top center',
            textfont=dict(color='white', size=10),
            name=nt, showlegend=False,
            hovertemplate=f'<b>{s}</b><br>Type: {nt}<extra></extra>'))
    # Legend entries
    for nt, col in TYPE_COLOR.items():
        fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
            marker=dict(size=12, color=col),
            name=nt.capitalize()))
    if highlight_gt:
        fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
            line=dict(color='#00D4FF', width=3), name='Correct edge'))
        fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
            line=dict(color='#FF6B6B', width=3), name='Extra edge'))
    fig.update_layout(
        template='plotly_dark', height=460,
        title=dict(text=title, font=dict(size=16)),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        margin=dict(l=20, r=20, t=60, b=20))
    fig.show()

draw_graph(GT, '<b>Ground Truth Physical Causal Graph</b>')

In [ ]:
draw_graph(W_thresh,
    '<b>VULGARIS Learned Causal Graph</b><br>'
    + '<span style="color:#00D4FF">Blue = matches GT</span>  '
    + '<span style="color:#FF6B6B">Red = extra edge</span>',
    highlight_gt=True)

## 6. Edge Precision / Recall vs Ground Truth

In [ ]:
# Binarise learned graph
learned_bin = (W_thresh > 0).astype(int)
gt_bin      = GT.astype(int)

np.fill_diagonal(learned_bin, 0)
np.fill_diagonal(gt_bin, 0)

TP = int((learned_bin * gt_bin).sum())
FP = int((learned_bin * (1 - gt_bin)).sum())
FN = int(((1 - learned_bin) * gt_bin).sum())

precision = TP / (TP + FP + 1e-8)
recall    = TP / (TP + FN + 1e-8)
f1        = 2 * precision * recall / (precision + recall + 1e-8)

print(f'True  Positive edges : {TP}')
print(f'False Positive edges : {FP}')
print(f'False Negative edges : {FN}')
print(f'Precision : {precision:.3f}')
print(f'Recall    : {recall:.3f}')
print(f'F1        : {f1:.3f}')

# Heatmap of edge weights
fig = go.Figure(go.Heatmap(
    z=W_eff.tolist(), x=SENSORS, y=SENSORS,
    colorscale='Blues', colorbar=dict(title='|W_eff|')))
fig.update_layout(template='plotly_dark', height=420,
    title='<b>Learned CRG Edge Weights |W_eff|</b>',
    xaxis_title='Effect', yaxis_title='Cause')
fig.show()

## 7. Failure Propagation Demo

**Scenario:** Pump P101 (Stage 1) fails.

Expected cascade: `P101 → FIT201 → LIT201 → P201 → FIT301 → LIT301 → P301 → ...`

VULGARIS BFS-propagates the fault through the **learned** causal graph
and returns a failure probability for every downstream node.

In [ ]:
# Trigger failure from P101 (index 2)
p101_idx = SENSORS.index('P101')

# propagate_failure returns list of (node_idx, prob) sorted descending
# excluding the trigger node itself
if hasattr(model.crg, 'propagate_failure'):
    result_list = model.crg.propagate_failure(
        triggered_nodes=[p101_idx], max_hops=6, decay=0.80)
    # Convert to dict (add trigger node back)
    fault_probs = {p101_idx: 1.0}
    for node_idx, prob in result_list:
        fault_probs[node_idx] = prob
else:
    # BFS fallback
    fault_probs = {p101_idx: 1.0}
    queue, visited = [(p101_idx, 1.0)], {p101_idx}
    while queue:
        node, prob = queue.pop(0)
        for j in range(N):
            if j not in visited and W_eff[node, j] > k_thresh/2:
                cp = prob * float(W_eff[node, j]) * 0.80
                if cp > 0.01:
                    fault_probs[j] = max(fault_probs.get(j, 0), cp)
                    visited.add(j); queue.append((j, cp))

print('Failure probabilities after P101 fault:')
for idx, prob in sorted(fault_probs.items(), key=lambda x: -x[1]):
    bar = '|' * int(prob * 30)
    print(f'  {SENSORS[idx]:8s}  {bar:<30s}  {prob:.3f}')

In [ ]:
# Visualise failure propagation on the graph
fig = go.Figure()

# All edges (faded)
for i in range(N):
    for j in range(N):
        if W_thresh[i, j] > 0:
            x0,y0 = POS[SENSORS[i]]; x1,y1 = POS[SENSORS[j]]
            fig.add_trace(go.Scatter(
                x=[x0,x1,None], y=[y0,y1,None], mode='lines',
                line=dict(color='rgba(100,100,100,0.3)', width=1),
                hoverinfo='skip', showlegend=False))

# Fault propagation edges (highlighted)
fault_set = set(fault_probs.keys())
for i in fault_set:
    for j in fault_set:
        if W_thresh[i,j] > 0 and i != j:
            x0,y0 = POS[SENSORS[i]]; x1,y1 = POS[SENSORS[j]]
            prob   = fault_probs.get(j, 0)
            fig.add_trace(go.Scatter(
                x=[x0,x1,None], y=[y0,y1,None], mode='lines',
                line=dict(color='#FF6B6B', width=2+prob*6),
                hoverinfo='skip', showlegend=False))

# Nodes coloured by failure probability
for s in SENSORS:
    idx  = SENSORS.index(s)
    prob = fault_probs.get(idx, 0.0)
    x,y  = POS[s]
    if idx == p101_idx:
        color = '#FF0000'; size = 38; label = s + ' FAULT'
    elif prob > 0:
        g = int(255 * (1 - prob))
        color = f'rgb(255,{g},0)'; size = 26 + int(prob*14)
        label = s + f' {prob:.2f}'
    else:
        color = '#51CF66'; size = 24; label = s
    fig.add_trace(go.Scatter(
        x=[x], y=[y], mode='markers+text',
        marker=dict(size=size, color=color, line=dict(color='white',width=1.5)),
        text=[label], textposition='top center',
        textfont=dict(color='white', size=9),
        hovertemplate=f'<b>{s}</b><br>Failure prob: {prob:.3f}<extra></extra>',
        showlegend=False))

fig.update_layout(
    template='plotly_dark', height=500,
    title='<b>Failure Propagation: P101 Fault Cascade</b><br>'
          '<span style="color:#FF0000">Red = fault origin  </span>'
          '<span style="color:#FF6B6B">Orange = affected  </span>'
          '<span style="color:#51CF66">Green = unaffected</span>',
    xaxis=dict(showgrid=False,zeroline=False,showticklabels=False),
    yaxis=dict(showgrid=False,zeroline=False,showticklabels=False),
    margin=dict(l=20,r=20,t=80,b=20))
fig.show()

## 8. Summary Dashboard

In [ ]:
# Results table
affected = sorted([(SENSORS[k],round(v,3)) for k,v in fault_probs.items()],
                  key=lambda x: -x[1])

fig_t = go.Figure(go.Table(
    header=dict(
        values=['Metric','Value'],
        fill_color='#1A2438', font=dict(color='white',size=13),
        align='center', height=32),
    cells=dict(
        values=[
            ['Sensors','GT edges','Learned edges',
             'Edge Precision','Edge Recall','Edge F1',
             'Fault origin','Nodes affected'],
            [str(N), str(int(GT.sum())),
             str(int((W_thresh>0).sum())),
             f'{precision:.3f}', f'{recall:.3f}', f'{f1:.3f}',
             'P101 (pump stage 1)',
             str(len(fault_probs)-1) + ' downstream'],
        ],
        fill_color=[['#0D1B2A']*8],
        font=dict(color=['white','#00D4FF'],size=12),
        align=['left','center'], height=28)))
fig_t.update_layout(template='plotly_dark', height=320,
    title=f'<b>VULGARIS v{vulgaris.__version__} — Causal Graph Results</b>')
fig_t.show()

print('Affected sensors in fault cascade:')
for name, prob in affected:
    print(f'  {name:8s}  {prob:.3f}')
print()
print('DLinear / LSTM: cannot discover causal structure or propagate faults.')
print('VULGARIS: learns the DAG and traces fault cascades automatically.')